# Binary & Multi-Class Classification

## Chest X-Ray Images (Pneumonia)

Importing all required dataset, packages & libraries

In [7]:
import pandas as pd
import numpy as np
import joblib
import os
from constants import RANDOM_SEED
from utils import set_seed
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import Subset

Since the images are of variable sizes, it is very important to scale them down to a fixed size. We will try out 512x512 and then 128x128 and if the difference isn't much, we can scale it down further.

In [8]:
from PIL import Image
import os

sample_dir = "D:/CitrusBits/pythonic-rebirth/datasets/chest_xray/train/NORMAL"
sample_files = os.listdir(sample_dir)[:5]  # peek at first 5

for f in sample_files:
    img_path = os.path.join(sample_dir, f)
    img = Image.open(img_path)
    print(f"{f}: size={img.size}, mode={img.mode}")

IM-0115-0001.jpeg: size=(2090, 1858), mode=L
IM-0117-0001.jpeg: size=(1422, 1152), mode=L
IM-0119-0001.jpeg: size=(1810, 1434), mode=L
IM-0122-0001.jpeg: size=(1618, 1279), mode=L
IM-0125-0001.jpeg: size=(1600, 1125), mode=L


In [9]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

DATA_DIR = "D:/CitrusBits/pythonic-rebirth/datasets/chest_xray"

# Transforms
IMG_SIZE = 224

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# datasets
train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transforms)
test_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "test"), transform=eval_transforms)
val_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), transform=eval_transforms)

print("Classes:", train_dataset.classes)
print("Class-to-index mapping:", train_dataset.class_to_idx)
print(f"Train size: {len(train_dataset)}")
print(f"Test size:  {len(test_dataset)}")
print(f"Val size:   {len(val_dataset)}")

# ============================================
# Stratified sampling of training data only
# ============================================
import numpy as np
from torch.utils.data import Subset

SAMPLE_FRACTION = 0.5  # 20% of training data

targets = np.array(train_dataset.targets)
normal_indices = np.where(targets == 0)[0]
pneumonia_indices = np.where(targets == 1)[0]

np.random.seed(RANDOM_SEED)
n_normal_sample = int(len(normal_indices) * SAMPLE_FRACTION)
n_pneumonia_sample = int(len(pneumonia_indices) * SAMPLE_FRACTION)

sampled_normal = np.random.choice(normal_indices, n_normal_sample, replace=False)
sampled_pneumonia = np.random.choice(pneumonia_indices, n_pneumonia_sample, replace=False)

sample_indices = np.concatenate([sampled_normal, sampled_pneumonia])
np.random.shuffle(sample_indices)

train_dataset_sampled = Subset(train_dataset, sample_indices)

print(f"Sampled train size: {len(train_dataset_sampled)}")
print(f"  NORMAL: {n_normal_sample}, PNEUMONIA: {n_pneumonia_sample}")
# ============================================

# dataloader (like delivery truck)
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset_sampled, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0)  # CHANGED: train_dataset -> train_dataset_sampled
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Check class imbalance in train set
from collections import Counter

train_labels = [train_dataset.targets[i] for i in
                sample_indices]  # CHANGED: now reflects the sampled subset, not full train_dataset
print("Train class distribution:", Counter(train_labels))

Classes: ['NORMAL', 'PNEUMONIA']
Class-to-index mapping: {'NORMAL': 0, 'PNEUMONIA': 1}
Train size: 5216
Test size:  624
Val size:   16
Sampled train size: 2607
  NORMAL: 670, PNEUMONIA: 1937
Train class distribution: Counter({1: 1937, 0: 670})


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import os
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")


# CNN Architecture

class ChestXrayCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(ChestXrayCNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 112 -> 56

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 56 -> 28

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 28 -> 14

        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  # 14x14 -> 1x1, avoids a huge flatten
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = ChestXrayCNN(num_classes=2).to(device)

# Handle class imbalance (2.9:1 Pneumonia:Normal)
class_counts = Counter(train_labels)  # from your earlier code: {1: 3875, 0: 1341}
total = sum(class_counts.values())
class_weights = torch.tensor(
    [total / class_counts[0], total / class_counts[1]],
    dtype=torch.float32
).to(device)
print(f"Class weights: {class_weights}")  # weights NORMAL higher since it's the minority

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# save or load setup
MODEL_DIR = "D:/CitrusBits/pythonic-rebirth/models"
os.makedirs(MODEL_DIR, exist_ok=True)
CNN_PATH = os.path.join(MODEL_DIR, "chest_xray_cnn_sscratch.pth")

EPOCHS = 6

if os.path.exists(CNN_PATH):
    print(f"Found existing model at {CNN_PATH}, loading instead of retraining...")
    model.load_state_dict(torch.load(CNN_PATH, map_location=device))
else:
    print("No saved model found, training CNN from scratch...")
    model.train()
    for epoch in range(EPOCHS):

        running_loss = 0.0
        correct = 0
        total_samples = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

        epoch_loss = running_loss / total_samples
        epoch_acc = correct / total_samples

        # --- NEW: ---
        # per-epoch test tracking
        model.eval()
        test_correct = 0
        test_total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                test_correct += (predicted == labels).sum().item()
                test_total += labels.size(0)
        test_acc = test_correct / test_total
        # --- END NEW ---

        print(f"Epoch {epoch + 1}/{EPOCHS} — Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f}")

    torch.save(model.state_dict(), CNN_PATH)
    print(f"Model saved to {CNN_PATH}")

Using device: cpu
Class weights: tensor([3.8910, 1.3459])
No saved model found, training CNN from scratch...
Epoch 1/6 — Loss: 0.4516, Train Acc: 0.8017
Epoch 2/6 — Loss: 0.2992, Train Acc: 0.8638
Epoch 3/6 — Loss: 0.2940, Train Acc: 0.8680
Epoch 4/6 — Loss: 0.2680, Train Acc: 0.8807
Epoch 5/6 — Loss: 0.2553, Train Acc: 0.8895
Epoch 6/6 — Loss: 0.2417, Train Acc: 0.8930
Model saved to D:/CitrusBits/pythonic-rebirth/models\chest_xray_cnn_sscratch.pth


In [11]:
model.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = correct / total
print(f"Test Accuracy: {test_acc:.4f}")

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(all_labels, all_preds, target_names=['NORMAL', 'PNEUMONIA']))
print(confusion_matrix(all_labels, all_preds))

Test Accuracy: 0.8237
              precision    recall  f1-score   support

      NORMAL       0.81      0.69      0.75       234
   PNEUMONIA       0.83      0.91      0.87       390

    accuracy                           0.82       624
   macro avg       0.82      0.80      0.81       624
weighted avg       0.82      0.82      0.82       624

[[161  73]
 [ 37 353]]


In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "D:/CitrusBits/pythonic-rebirth/datasets/chest_xray"
IMG_SIZE_PRETRAINED = 224

# --- Transforms — same grayscale->RGB conversion as before, now at 224 for ResNet ---
train_transforms_pt = transforms.Compose([
    transforms.Resize((IMG_SIZE_PRETRAINED, IMG_SIZE_PRETRAINED)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transforms_pt = transforms.Compose([
    transforms.Resize((IMG_SIZE_PRETRAINED, IMG_SIZE_PRETRAINED)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# --- Datasets — reuse the same sampling logic as your from-scratch run ---
full_train_dataset_pt = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transforms_pt)
test_dataset_pt = datasets.ImageFolder(os.path.join(DATA_DIR, "test"), transform=eval_transforms_pt)

import numpy as np
from torch.utils.data import Subset
from constants import RANDOM_SEED

SAMPLE_FRACTION = 0.5  # matching your best from-scratch run
targets = np.array(full_train_dataset_pt.targets)
normal_indices = np.where(targets == 0)[0]
pneumonia_indices = np.where(targets == 1)[0]

np.random.seed(RANDOM_SEED)
n_normal = int(len(normal_indices) * SAMPLE_FRACTION)
n_pneumonia = int(len(pneumonia_indices) * SAMPLE_FRACTION)
sampled_normal = np.random.choice(normal_indices, n_normal, replace=False)
sampled_pneumonia = np.random.choice(pneumonia_indices, n_pneumonia, replace=False)
sample_indices = np.concatenate([sampled_normal, sampled_pneumonia])
np.random.shuffle(sample_indices)

train_dataset_pt = Subset(full_train_dataset_pt, sample_indices)

BATCH_SIZE = 32
train_loader_pt = DataLoader(train_dataset_pt, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader_pt = DataLoader(test_dataset_pt, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

train_labels_pt = [full_train_dataset_pt.targets[i] for i in sample_indices]
print("Train class distribution:", Counter(train_labels_pt))

# --- Pretrained ResNet18, frozen backbone ---
resnet_xray = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

for param in resnet_xray.parameters():
    param.requires_grad = False

num_features = resnet_xray.fc.in_features
resnet_xray.fc = nn.Linear(num_features, 2)  # binary: NORMAL vs PNEUMONIA
resnet_xray = resnet_xray.to(device)

# --- Class weights still needed — X-Ray remains imbalanced (2.9:1) ---
from collections import Counter

class_counts = Counter(train_labels_pt)
total = sum(class_counts.values())
class_weights = torch.tensor(
    [total / class_counts[0], total / class_counts[1]],
    dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(resnet_xray.fc.parameters(), lr=0.0001)  # only the new fc layer trains

# --- Save/load ---
MODEL_DIR = "D:/CitrusBits/pythonic-rebirth/models"
RESNET_XRAY_PATH = os.path.join(MODEL_DIR, "chest_xray_resnet18_transfer.pth")
EPOCHS = 6

if os.path.exists(RESNET_XRAY_PATH):
    print(f"Found existing model, loading instead of retraining...")
    resnet_xray.load_state_dict(torch.load(RESNET_XRAY_PATH, map_location=device))
else:
    print("No saved model found, training transfer learning model...")
    for epoch in range(EPOCHS):
        resnet_xray.train()
        running_loss, correct, total_samples = 0.0, 0, 0

        for images, labels in train_loader_pt:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = resnet_xray(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

        epoch_loss = running_loss / total_samples
        epoch_acc = correct / total_samples

        resnet_xray.eval()
        test_correct, test_total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader_pt:
                images, labels = images.to(device), labels.to(device)
                outputs = resnet_xray(images)
                _, predicted = torch.max(outputs, 1)
                test_correct += (predicted == labels).sum().item()
                test_total += labels.size(0)
        test_acc = test_correct / test_total

        print(
            f"Epoch {epoch + 1}/{EPOCHS} — Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f}, Test Acc: {test_acc:.4f}")

    torch.save(resnet_xray.state_dict(), RESNET_XRAY_PATH)
    print(f"Model saved to {RESNET_XRAY_PATH}")

Train class distribution: Counter({1: 1937, 0: 670})
No saved model found, training transfer learning model...
Epoch 1/6 — Train Loss: 0.5948, Train Acc: 0.6870, Test Acc: 0.8077
Epoch 2/6 — Train Loss: 0.4447, Train Acc: 0.8393, Test Acc: 0.8494
Epoch 3/6 — Train Loss: 0.3685, Train Acc: 0.8761, Test Acc: 0.8606
Epoch 4/6 — Train Loss: 0.3242, Train Acc: 0.8914, Test Acc: 0.8622
Epoch 5/6 — Train Loss: 0.2914, Train Acc: 0.8999, Test Acc: 0.8734
Epoch 6/6 — Train Loss: 0.2780, Train Acc: 0.8972, Test Acc: 0.8750
Model saved to D:/CitrusBits/pythonic-rebirth/models\chest_xray_resnet18_transfer.pth


In [13]:
from sklearn.metrics import classification_report, confusion_matrix

resnet_xray.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader_pt:
        images, labels = images.to(device), labels.to(device)
        outputs = resnet_xray(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = correct / total
print(f"Test Accuracy: {test_acc:.4f}")

print(classification_report(all_labels, all_preds, target_names=['NORMAL', 'PNEUMONIA']))
print(confusion_matrix(all_labels, all_preds))

Test Accuracy: 0.8750
              precision    recall  f1-score   support

      NORMAL       0.89      0.76      0.82       234
   PNEUMONIA       0.87      0.95      0.90       390

    accuracy                           0.88       624
   macro avg       0.88      0.85      0.86       624
weighted avg       0.88      0.88      0.87       624

[[177  57]
 [ 21 369]]
